# Public research notebook

This notebook is a cleaned public version of the original
research workflow.

## Execution model

- All filesystem paths are relative to the repository root.
- No external mounted filesystem is required.
- Stored cell outputs have been removed.
- Generated files are written below the local `results/`
  directory.
- The archival source notebook remains unchanged.


In [ ]:
# Portable repository configuration
#
# The notebook assumes that it is executed from the repository
# root or from a cloned copy of the repository.

from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()

# Move upward when the notebook is launched from a nested folder.
if REPOSITORY_ROOT.name in {
    "lorenz",
    "rossler",
    "duffing",
    "kuramoto",
    "stuart_landau",
    "coupled_map_lattice",
}:
    REPOSITORY_ROOT = REPOSITORY_ROOT.parents[1]

DATA_DIR = REPOSITORY_ROOT / "data"
RESULTS_DIR = REPOSITORY_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"

for directory in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    CHECKPOINTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Results directory: {RESULTS_DIR}")


In [ ]:
# ============================================================
# KURAMOTO REORGANIZATION WINDOW v1 — CHECKPOINT
# phase-only validation after Stuart–Landau
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# ============================================================
# OUTPUT
# ============================================================

ROOT = "notebooks/Kuramoto"
OUT = ROOT + "/kuramoto_reorganization_window_v1_CHECKPOINT"
os.makedirs(OUT, exist_ok=True)

CHECKPOINT = OUT + "/kuramoto_reorganization_all_checkpoint.csv"

print("Saving to:", OUT)

# ============================================================
# PARAMETERS
# ============================================================

N = 400
SEEDS = list(range(20))

K_VALUES = np.round(np.arange(0.1, 2.001, 0.05), 3)

omega_std = 0.4

dt = 0.03
steps = 4000
discard = 1500

cluster_threshold = 0.15

# ============================================================
# HELPERS
# ============================================================

def order_parameter(theta):
    return float(np.abs(np.mean(np.exp(1j * theta))))

def count_clusters(theta, threshold=0.15):
    th = np.sort(np.mod(theta, 2*np.pi))
    gaps = np.diff(th)
    circular_gap = (th[0] + 2*np.pi) - th[-1]
    gaps = np.append(gaps, circular_gap)
    return int(np.sum(gaps > threshold))

def memory_score(cluster_series, window=200):
    s = pd.Series(cluster_series)
    return float((s.rolling(window).std().fillna(0) < 0.2).mean())

# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT):
    all_df = pd.read_csv(CHECKPOINT)
    rows = all_df.to_dict("records")
    done = set(zip(all_df["K"].round(3), all_df["seed"]))
    print("Loaded checkpoint rows:", len(rows))
else:
    rows = []
    done = set()
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for K in K_VALUES:

    print(f"\nK={K}")

    for seed in tqdm(SEEDS):

        key = (round(float(K), 3), int(seed))

        if key in done:
            continue

        rng = np.random.default_rng(seed)

        omega = rng.normal(0, omega_std, N)
        theta = rng.uniform(-np.pi, np.pi, N)

        R_series = []
        cluster_series = []

        for t in range(steps):

            # mean-field Kuramoto coupling
            Z = np.mean(np.exp(1j * theta))
            R = np.abs(Z)
            psi = np.angle(Z)

            dtheta = omega + K * R * np.sin(psi - theta)

            theta = theta + dt * dtheta

            if t >= discard:
                R_now = order_parameter(theta)
                clusters = count_clusters(theta, cluster_threshold)

                R_series.append(R_now)
                cluster_series.append(clusters)

        R_series = np.array(R_series)
        cluster_series = np.array(cluster_series)

        dominant_cluster = pd.Series(cluster_series).mode().iloc[0]
        dominant_fraction = float(np.mean(cluster_series == dominant_cluster))

        switching_rate = float(np.sum(np.diff(cluster_series) != 0) / len(cluster_series))

        reorganization_score = float(
            np.std(cluster_series)
            * (1 - dominant_fraction)
            * (switching_rate + 1e-9)
        )

        row = {
            "K": K,
            "seed": seed,

            "R_mean": float(np.mean(R_series)),
            "R_std": float(np.std(R_series)),

            "cluster_mean": float(np.mean(cluster_series)),
            "cluster_std": float(np.std(cluster_series)),

            "dominant_cluster": int(dominant_cluster),
            "dominant_fraction": dominant_fraction,
            "switching_rate": switching_rate,
            "memory_score": memory_score(cluster_series),

            "reorganization_score": reorganization_score
        }

        rows.append(row)
        done.add(key)

        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

print("\nFULL LOOP DONE")

# ============================================================
# AGGREGATE
# ============================================================

all_df = pd.DataFrame(rows)

agg = (
    all_df
    .groupby("K")
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"],
        "cluster_mean": ["mean", "std"],
        "cluster_std": ["mean", "std"],
        "dominant_fraction": ["mean", "std"],
        "switching_rate": ["mean", "std"],
        "memory_score": ["mean", "std"],
        "reorganization_score": ["mean", "std"],
    })
)

agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()

peak_idx = agg["reorganization_score_mean"].idxmax()

summary = pd.DataFrame([{
    "peak_K_reorganization": agg.loc[peak_idx, "K"],
    "peak_reorganization_score": agg.loc[peak_idx, "reorganization_score_mean"],
    "R_mean_at_peak": agg.loc[peak_idx, "R_mean_mean"],
    "cluster_mean_at_peak": agg.loc[peak_idx, "cluster_mean_mean"],
    "dominant_fraction_at_peak": agg.loc[peak_idx, "dominant_fraction_mean"],
    "switching_rate_at_peak": agg.loc[peak_idx, "switching_rate_mean"],
    "memory_score_at_peak": agg.loc[peak_idx, "memory_score_mean"],
}])

# ============================================================
# SAVE
# ============================================================

all_df.to_csv(OUT + "/kuramoto_reorganization_all.csv", index=False)
agg.to_csv(OUT + "/kuramoto_reorganization_aggregate.csv", index=False)
summary.to_csv(OUT + "/kuramoto_reorganization_summary.csv", index=False)

# ============================================================
# FIGURES
# ============================================================

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["R_mean_mean"], marker="o")
plt.xlabel("K")
plt.ylabel("R mean")
plt.title("Kuramoto: synchronization vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_R_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["cluster_mean_mean"], marker="o")
plt.xlabel("K")
plt.ylabel("Cluster mean")
plt.title("Kuramoto: cluster count vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_cluster_mean_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["switching_rate_mean"], marker="o")
plt.xlabel("K")
plt.ylabel("Switching rate")
plt.title("Kuramoto: switching rate vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_switching_rate_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["reorganization_score_mean"], marker="o")
plt.xlabel("K")
plt.ylabel("Reorganization score")
plt.title("Kuramoto: reorganization window")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_reorganization_score.png", dpi=300)
plt.show()

# ============================================================
# DISPLAY
# ============================================================

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE PREVIEW")
display(agg.head(30))

print("\nSaved to:")
print(OUT)

In [ ]:
# ============================================================
# KURAMOTO TOPOLOGY ROBUSTNESS v1 — CHECKPOINT
# phase-only topology comparison
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import networkx as nx
import os

ROOT = "notebooks/Kuramoto"
OUT = ROOT + "/kuramoto_topology_robustness_v1_CHECKPOINT"
os.makedirs(OUT, exist_ok=True)

CHECKPOINT = OUT + "/kuramoto_topology_all_checkpoint.csv"

print("Saving to:", OUT)

# ============================================================
# PARAMETERS
# ============================================================

N = 400
SEEDS = list(range(20))

K_VALUES = np.round(np.arange(0.2, 2.201, 0.1), 3)

omega_std = 0.4

dt = 0.03
steps = 3500
discard = 1200

cluster_threshold = 0.15

TOPOLOGIES = [
    "all_to_all",
    "small_world",
    "scale_free",
    "modular",
    "spatial_lattice"
]

# ============================================================
# HELPERS
# ============================================================

def order_parameter(theta):
    return float(np.abs(np.mean(np.exp(1j * theta))))

def count_clusters(theta, threshold=0.15):
    th = np.sort(np.mod(theta, 2*np.pi))
    gaps = np.diff(th)
    circular_gap = (th[0] + 2*np.pi) - th[-1]
    gaps = np.append(gaps, circular_gap)
    return int(np.sum(gaps > threshold))

def memory_score(cluster_series, window=200):
    s = pd.Series(cluster_series)
    return float((s.rolling(window).std().fillna(0) < 0.2).mean())

def make_topology(name, N, seed):
    rng = np.random.default_rng(seed)

    if name == "all_to_all":
        A = np.ones((N, N)) - np.eye(N)

    elif name == "small_world":
        G = nx.watts_strogatz_graph(N, k=8, p=0.12, seed=seed)
        A = nx.to_numpy_array(G)

    elif name == "scale_free":
        G = nx.barabasi_albert_graph(N, m=4, seed=seed)
        A = nx.to_numpy_array(G)

    elif name == "modular":
        sizes = [N // 4, N // 4, N // 4, N - 3 * (N // 4)]
        p_in = 0.08
        p_out = 0.008
        probs = [[p_in if i == j else p_out for j in range(4)] for i in range(4)]
        G = nx.stochastic_block_model(sizes, probs, seed=seed)
        A = nx.to_numpy_array(G)

    elif name == "spatial_lattice":
        side = int(np.sqrt(N))
        if side * side != N:
            raise ValueError("N must be a square for spatial_lattice")
        G = nx.grid_2d_graph(side, side, periodic=True)
        G = nx.convert_node_labels_to_integers(G)
        A = nx.to_numpy_array(G)

    else:
        raise ValueError("Unknown topology")

    np.fill_diagonal(A, 0)
    deg = A.sum(axis=1)
    deg[deg == 0] = 1

    return A, deg

# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT):
    all_df = pd.read_csv(CHECKPOINT)
    rows = all_df.to_dict("records")
    done = set(zip(
        all_df["topology"],
        all_df["K"].round(3),
        all_df["seed"]
    ))
    print("Loaded checkpoint rows:", len(rows))
else:
    rows = []
    done = set()
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for topology in TOPOLOGIES:

    print("\n" + "="*70)
    print("TOPOLOGY:", topology)
    print("="*70)

    for K in K_VALUES:

        print(f"\nK={K}")

        for seed in tqdm(SEEDS):

            key = (topology, round(float(K), 3), int(seed))

            if key in done:
                continue

            rng = np.random.default_rng(seed)

            A, deg = make_topology(topology, N, seed)

            omega = rng.normal(0, omega_std, N)
            theta = rng.uniform(-np.pi, np.pi, N)

            R_series = []
            cluster_series = []

            for t in range(steps):

                sin_diff = np.sin(theta[None, :] - theta[:, None])
                coupling = (A * sin_diff).sum(axis=1) / deg

                dtheta = omega + K * coupling

                theta = theta + dt * dtheta

                if t >= discard:
                    R_series.append(order_parameter(theta))
                    cluster_series.append(count_clusters(theta, cluster_threshold))

            R_series = np.array(R_series)
            cluster_series = np.array(cluster_series)

            dominant_cluster = pd.Series(cluster_series).mode().iloc[0]
            dominant_fraction = float(np.mean(cluster_series == dominant_cluster))
            switching_rate = float(np.sum(np.diff(cluster_series) != 0) / len(cluster_series))

            reorganization_score = float(
                np.std(cluster_series)
                * (1 - dominant_fraction)
                * (switching_rate + 1e-9)
            )

            row = {
                "topology": topology,
                "K": K,
                "seed": seed,

                "R_mean": float(np.mean(R_series)),
                "R_std": float(np.std(R_series)),

                "cluster_mean": float(np.mean(cluster_series)),
                "cluster_std": float(np.std(cluster_series)),

                "dominant_cluster": int(dominant_cluster),
                "dominant_fraction": dominant_fraction,
                "switching_rate": switching_rate,
                "memory_score": memory_score(cluster_series),

                "reorganization_score": reorganization_score
            }

            rows.append(row)
            done.add(key)

            pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

print("\nFULL LOOP DONE")

# ============================================================
# AGGREGATE
# ============================================================

all_df = pd.DataFrame(rows)

agg = (
    all_df
    .groupby(["topology", "K"])
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"],
        "cluster_mean": ["mean", "std"],
        "cluster_std": ["mean", "std"],
        "dominant_fraction": ["mean", "std"],
        "switching_rate": ["mean", "std"],
        "memory_score": ["mean", "std"],
        "reorganization_score": ["mean", "std"],
    })
)

agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()

summary_rows = []

for topology in TOPOLOGIES:
    sub = agg[agg["topology"] == topology]
    idx = sub["reorganization_score_mean"].idxmax()

    summary_rows.append({
        "topology": topology,
        "peak_K_reorganization": sub.loc[idx, "K"],
        "peak_reorganization_score": sub.loc[idx, "reorganization_score_mean"],
        "R_mean_at_peak": sub.loc[idx, "R_mean_mean"],
        "cluster_mean_at_peak": sub.loc[idx, "cluster_mean_mean"],
        "dominant_fraction_at_peak": sub.loc[idx, "dominant_fraction_mean"],
        "switching_rate_at_peak": sub.loc[idx, "switching_rate_mean"],
        "memory_score_at_peak": sub.loc[idx, "memory_score_mean"],
    })

summary = pd.DataFrame(summary_rows)

# ============================================================
# SAVE
# ============================================================

all_df.to_csv(OUT + "/kuramoto_topology_all.csv", index=False)
agg.to_csv(OUT + "/kuramoto_topology_aggregate.csv", index=False)
summary.to_csv(OUT + "/kuramoto_topology_summary.csv", index=False)

# ============================================================
# FIGURES
# ============================================================

plt.figure(figsize=(10,6))
for topology in TOPOLOGIES:
    sub = agg[agg["topology"] == topology]
    plt.plot(sub["K"], sub["R_mean_mean"], marker="o", label=topology)
plt.xlabel("K")
plt.ylabel("R mean")
plt.title("Kuramoto topology: synchronization vs K")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_topology_R_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for topology in TOPOLOGIES:
    sub = agg[agg["topology"] == topology]
    plt.plot(sub["K"], sub["cluster_mean_mean"], marker="o", label=topology)
plt.xlabel("K")
plt.ylabel("Cluster mean")
plt.title("Kuramoto topology: cluster count vs K")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_topology_cluster_mean.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for topology in TOPOLOGIES:
    sub = agg[agg["topology"] == topology]
    plt.plot(sub["K"], sub["switching_rate_mean"], marker="o", label=topology)
plt.xlabel("K")
plt.ylabel("Switching rate")
plt.title("Kuramoto topology: switching rate vs K")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_topology_switching_rate.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for topology in TOPOLOGIES:
    sub = agg[agg["topology"] == topology]
    plt.plot(sub["K"], sub["reorganization_score_mean"], marker="o", label=topology)
plt.xlabel("K")
plt.ylabel("Reorganization score")
plt.title("Kuramoto topology: reorganization score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_topology_reorganization_score.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(summary["peak_K_reorganization"], summary["topology"], s=120)
plt.xlabel("Peak K")
plt.ylabel("Topology")
plt.title("Kuramoto topology: peak K summary")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_topology_peak_K_summary.png", dpi=300)
plt.show()

# ============================================================
# DISPLAY
# ============================================================

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE PREVIEW")
display(agg.head(30))

print("\nSaved to:")
print(OUT)

In [ ]:
# ============================================================
# KURAMOTO ADAPTIVE COUPLING v1 — CHECKPOINT
# phase-only adaptive coupling test
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

ROOT = "notebooks/Kuramoto"
OUT = ROOT + "/kuramoto_adaptive_coupling_v1_CHECKPOINT"
os.makedirs(OUT, exist_ok=True)

CHECKPOINT = OUT + "/kuramoto_adaptive_all_checkpoint.csv"

print("Saving to:", OUT)

# ============================================================
# PARAMETERS
# ============================================================

N = 400
SEEDS = list(range(20))

K_VALUES = np.round(np.arange(0.2, 2.201, 0.1), 3)

omega_std = 0.4

dt = 0.03
steps = 3500
discard = 1200

cluster_threshold = 0.15

# adaptive coupling parameters
eta = 0.015
decay = 0.01
Kij_min = 0.0
Kij_max = 3.0
strong_threshold = 1.5
weak_threshold = 0.2

# ============================================================
# HELPERS
# ============================================================

def order_parameter(theta):
    return float(np.abs(np.mean(np.exp(1j * theta))))

def count_clusters(theta, threshold=0.15):
    th = np.sort(np.mod(theta, 2*np.pi))
    gaps = np.diff(th)
    circular_gap = (th[0] + 2*np.pi) - th[-1]
    gaps = np.append(gaps, circular_gap)
    return int(np.sum(gaps > threshold))

def memory_score(cluster_series, window=200):
    s = pd.Series(cluster_series)
    return float((s.rolling(window).std().fillna(0) < 0.2).mean())

# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT):
    all_df = pd.read_csv(CHECKPOINT)
    rows = all_df.to_dict("records")
    done = set(zip(all_df["K_base"].round(3), all_df["seed"]))
    print("Loaded checkpoint rows:", len(rows))
else:
    rows = []
    done = set()
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for K_base in K_VALUES:

    print(f"\nK_base={K_base}")

    for seed in tqdm(SEEDS):

        key = (round(float(K_base), 3), int(seed))

        if key in done:
            continue

        rng = np.random.default_rng(seed)

        omega = rng.normal(0, omega_std, N)
        theta = rng.uniform(-np.pi, np.pi, N)

        # adaptive pairwise coupling matrix
        Kij = np.ones((N, N)) * K_base
        np.fill_diagonal(Kij, 0.0)

        R_series = []
        cluster_series = []

        for t in range(steps):

            phase_diff = theta[None, :] - theta[:, None]

            # dynamics
            coupling = (Kij * np.sin(phase_diff)).sum(axis=1) / (N - 1)
            dtheta = omega + coupling
            theta = theta + dt * dtheta

            # adaptation: compatible phases strengthen, incompatible weaken
            compatibility = np.cos(phase_diff)
            dK = eta * compatibility - decay * (Kij - K_base)
            Kij = Kij + dt * dK
            Kij = np.clip(Kij, Kij_min, Kij_max)
            np.fill_diagonal(Kij, 0.0)

            if t >= discard:
                R_series.append(order_parameter(theta))
                cluster_series.append(count_clusters(theta, cluster_threshold))

        R_series = np.array(R_series)
        cluster_series = np.array(cluster_series)

        dominant_cluster = pd.Series(cluster_series).mode().iloc[0]
        dominant_fraction = float(np.mean(cluster_series == dominant_cluster))
        switching_rate = float(np.sum(np.diff(cluster_series) != 0) / len(cluster_series))

        Kij_nonzero = Kij[Kij > 0]

        reorganization_score = float(
            np.std(cluster_series)
            * (1 - dominant_fraction)
            * (switching_rate + 1e-9)
        )

        adaptive_network_score = float(
            np.std(Kij_nonzero)
            * np.mean(Kij_nonzero)
        )

        adaptive_reorganization_score = float(
            reorganization_score * (1 + adaptive_network_score)
        )

        row = {
            "K_base": K_base,
            "seed": seed,

            "R_mean": float(np.mean(R_series)),
            "R_std": float(np.std(R_series)),

            "cluster_mean": float(np.mean(cluster_series)),
            "cluster_std": float(np.std(cluster_series)),

            "dominant_cluster": int(dominant_cluster),
            "dominant_fraction": dominant_fraction,
            "switching_rate": switching_rate,
            "memory_score": memory_score(cluster_series),

            "Kij_mean_final": float(np.mean(Kij_nonzero)),
            "Kij_std_final": float(np.std(Kij_nonzero)),
            "Kij_density_weak": float(np.mean(Kij_nonzero < weak_threshold)),
            "Kij_density_strong": float(np.mean(Kij_nonzero > strong_threshold)),

            "reorganization_score": reorganization_score,
            "adaptive_network_score": adaptive_network_score,
            "adaptive_reorganization_score": adaptive_reorganization_score
        }

        rows.append(row)
        done.add(key)

        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

print("\nFULL LOOP DONE")

# ============================================================
# AGGREGATE
# ============================================================

all_df = pd.DataFrame(rows)

agg = (
    all_df
    .groupby("K_base")
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"],
        "cluster_mean": ["mean", "std"],
        "cluster_std": ["mean", "std"],
        "dominant_fraction": ["mean", "std"],
        "switching_rate": ["mean", "std"],
        "memory_score": ["mean", "std"],
        "Kij_mean_final": ["mean", "std"],
        "Kij_std_final": ["mean", "std"],
        "Kij_density_weak": ["mean", "std"],
        "Kij_density_strong": ["mean", "std"],
        "reorganization_score": ["mean", "std"],
        "adaptive_network_score": ["mean", "std"],
        "adaptive_reorganization_score": ["mean", "std"],
    })
)

agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()

peak_idx = agg["adaptive_reorganization_score_mean"].idxmax()

summary = pd.DataFrame([{
    "peak_K_adaptive_reorganization": agg.loc[peak_idx, "K_base"],
    "peak_adaptive_reorganization_score": agg.loc[peak_idx, "adaptive_reorganization_score_mean"],
    "raw_reorganization_score_at_peak": agg.loc[peak_idx, "reorganization_score_mean"],
    "adaptive_network_score_at_peak": agg.loc[peak_idx, "adaptive_network_score_mean"],
    "R_mean_at_peak": agg.loc[peak_idx, "R_mean_mean"],
    "cluster_mean_at_peak": agg.loc[peak_idx, "cluster_mean_mean"],
    "dominant_fraction_at_peak": agg.loc[peak_idx, "dominant_fraction_mean"],
    "switching_rate_at_peak": agg.loc[peak_idx, "switching_rate_mean"],
    "Kij_mean_final_at_peak": agg.loc[peak_idx, "Kij_mean_final_mean"],
    "Kij_std_final_at_peak": agg.loc[peak_idx, "Kij_std_final_mean"],
    "Kij_density_strong_at_peak": agg.loc[peak_idx, "Kij_density_strong_mean"],
}])

# ============================================================
# SAVE
# ============================================================

all_df.to_csv(OUT + "/kuramoto_adaptive_all.csv", index=False)
agg.to_csv(OUT + "/kuramoto_adaptive_aggregate.csv", index=False)
summary.to_csv(OUT + "/kuramoto_adaptive_summary.csv", index=False)

# ============================================================
# FIGURES
# ============================================================

plt.figure(figsize=(10,6))
plt.plot(agg["K_base"], agg["R_mean_mean"], marker="o")
plt.xlabel("K base")
plt.ylabel("R mean")
plt.title("Kuramoto adaptive coupling: synchronization")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_adaptive_R_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
plt.plot(agg["K_base"], agg["cluster_mean_mean"], marker="o")
plt.xlabel("K base")
plt.ylabel("Cluster mean")
plt.title("Kuramoto adaptive coupling: cluster count")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_adaptive_cluster_mean.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
plt.plot(agg["K_base"], agg["Kij_mean_final_mean"], marker="o", label="Kij mean")
plt.plot(agg["K_base"], agg["Kij_std_final_mean"], marker="o", label="Kij std")
plt.xlabel("K base")
plt.ylabel("Adaptive coupling statistics")
plt.title("Kuramoto adaptive coupling: final network structure")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_adaptive_Kij_structure.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
plt.plot(agg["K_base"], agg["reorganization_score_mean"], marker="o", label="raw reorganization")
plt.plot(agg["K_base"], agg["adaptive_reorganization_score_mean"], marker="o", label="adaptive reorganization")
plt.xlabel("K base")
plt.ylabel("Score")
plt.title("Kuramoto adaptive coupling: reorganization score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_adaptive_reorganization_score.png", dpi=300)
plt.show()

# ============================================================
# DISPLAY
# ============================================================

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE PREVIEW")
display(agg.head(30))

print("\nSaved to:")
print(OUT)

In [ ]:
# ============================================================
# KURAMOTO MINIMAL REPLICATION v1 — CHECKPOINT
# phase-only robustness replication
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

ROOT = "notebooks/Kuramoto"
OUT = ROOT + "/kuramoto_minimal_replication_v1_CHECKPOINT"
os.makedirs(OUT, exist_ok=True)

CHECKPOINT = OUT + "/kuramoto_minimal_replication_all_checkpoint.csv"

print("Saving to:", OUT)

# ============================================================
# PARAMETERS
# ============================================================

REPLICATIONS = [
    {"replication": "baseline_shifted_seeds", "N": 400, "dt": 0.03,  "omega_std": 0.4,  "seed_offset": 1000},
    {"replication": "lower_noise",           "N": 400, "dt": 0.03,  "omega_std": 0.35, "seed_offset": 2000},
    {"replication": "higher_noise",          "N": 400, "dt": 0.03,  "omega_std": 0.45, "seed_offset": 3000},
    {"replication": "smaller_dt",            "N": 400, "dt": 0.025, "omega_std": 0.4,  "seed_offset": 4000},
    {"replication": "smaller_N",             "N": 300, "dt": 0.03,  "omega_std": 0.4,  "seed_offset": 5000},
]

SEEDS = list(range(10))
K_VALUES = np.round(np.arange(0.45, 1.151, 0.025), 3)

steps = 3500
discard = 1200

cluster_threshold = 0.15

# ============================================================
# HELPERS
# ============================================================

def order_parameter(theta):
    return float(np.abs(np.mean(np.exp(1j * theta))))

def count_clusters(theta, threshold=0.15):
    th = np.sort(np.mod(theta, 2*np.pi))
    gaps = np.diff(th)
    circular_gap = (th[0] + 2*np.pi) - th[-1]
    gaps = np.append(gaps, circular_gap)
    return int(np.sum(gaps > threshold))

def memory_score(cluster_series, window=200):
    s = pd.Series(cluster_series)
    return float((s.rolling(window).std().fillna(0) < 0.2).mean())

# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT):
    all_df = pd.read_csv(CHECKPOINT)
    rows = all_df.to_dict("records")
    done = set(zip(
        all_df["replication"],
        all_df["K"].round(3),
        all_df["seed"]
    ))
    print("Loaded checkpoint rows:", len(rows))
else:
    rows = []
    done = set()
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for cfg in REPLICATIONS:

    rep_name = cfg["replication"]
    N = cfg["N"]
    dt = cfg["dt"]
    omega_std = cfg["omega_std"]
    seed_offset = cfg["seed_offset"]

    print("\n" + "="*70)
    print("REPLICATION:", rep_name, "| N:", N, "| dt:", dt, "| omega_std:", omega_std)
    print("="*70)

    for K in K_VALUES:

        print(f"\nK={K}")

        for seed in tqdm(SEEDS):

            actual_seed = seed + seed_offset
            key = (rep_name, round(float(K), 3), int(seed))

            if key in done:
                continue

            rng = np.random.default_rng(actual_seed)

            omega = rng.normal(0, omega_std, N)
            theta = rng.uniform(-np.pi, np.pi, N)

            R_series = []
            cluster_series = []

            for t in range(steps):

                Z = np.mean(np.exp(1j * theta))
                R = np.abs(Z)
                psi = np.angle(Z)

                dtheta = omega + K * R * np.sin(psi - theta)
                theta = theta + dt * dtheta

                if t >= discard:
                    R_series.append(order_parameter(theta))
                    cluster_series.append(count_clusters(theta, cluster_threshold))

            R_series = np.array(R_series)
            cluster_series = np.array(cluster_series)

            dominant_cluster = pd.Series(cluster_series).mode().iloc[0]
            dominant_fraction = float(np.mean(cluster_series == dominant_cluster))
            switching_rate = float(np.sum(np.diff(cluster_series) != 0) / len(cluster_series))

            reorganization_score = float(
                np.std(cluster_series)
                * (1 - dominant_fraction)
                * (switching_rate + 1e-9)
            )

            row = {
                "replication": rep_name,
                "N": N,
                "dt": dt,
                "omega_std": omega_std,
                "K": K,
                "seed": seed,
                "actual_seed": actual_seed,

                "R_mean": float(np.mean(R_series)),
                "R_std": float(np.std(R_series)),

                "cluster_mean": float(np.mean(cluster_series)),
                "cluster_std": float(np.std(cluster_series)),

                "dominant_cluster": int(dominant_cluster),
                "dominant_fraction": dominant_fraction,
                "switching_rate": switching_rate,
                "memory_score": memory_score(cluster_series),

                "reorganization_score": reorganization_score
            }

            rows.append(row)
            done.add(key)

            pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

print("\nFULL LOOP DONE")

# ============================================================
# AGGREGATE
# ============================================================

all_df = pd.DataFrame(rows)

agg = (
    all_df
    .groupby(["replication", "K"])
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"],
        "cluster_mean": ["mean", "std"],
        "cluster_std": ["mean", "std"],
        "dominant_fraction": ["mean", "std"],
        "switching_rate": ["mean", "std"],
        "memory_score": ["mean", "std"],
        "reorganization_score": ["mean", "std"],
    })
)

agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()

summary_rows = []

for rep_name in agg["replication"].unique():

    sub = agg[agg["replication"] == rep_name].copy()
    idx = sub["reorganization_score_mean"].idxmax()

    summary_rows.append({
        "replication": rep_name,
        "peak_K_reorganization": sub.loc[idx, "K"],
        "peak_reorganization_score": sub.loc[idx, "reorganization_score_mean"],
        "R_mean_at_peak": sub.loc[idx, "R_mean_mean"],
        "cluster_mean_at_peak": sub.loc[idx, "cluster_mean_mean"],
        "dominant_fraction_at_peak": sub.loc[idx, "dominant_fraction_mean"],
        "switching_rate_at_peak": sub.loc[idx, "switching_rate_mean"],
        "memory_score_at_peak": sub.loc[idx, "memory_score_mean"],
    })

summary = pd.DataFrame(summary_rows)

replication_window = pd.DataFrame([{
    "K_min_peak": summary["peak_K_reorganization"].min(),
    "K_max_peak": summary["peak_K_reorganization"].max(),
    "K_mean_peak": summary["peak_K_reorganization"].mean(),
    "K_median_peak": summary["peak_K_reorganization"].median(),
    "K_std_peak": summary["peak_K_reorganization"].std(),
    "n_replications": len(summary)
}])

# ============================================================
# SAVE
# ============================================================

all_df.to_csv(OUT + "/kuramoto_minimal_replication_all.csv", index=False)
agg.to_csv(OUT + "/kuramoto_minimal_replication_aggregate.csv", index=False)
summary.to_csv(OUT + "/kuramoto_minimal_replication_summary.csv", index=False)
replication_window.to_csv(OUT + "/kuramoto_minimal_replication_window.csv", index=False)

# ============================================================
# FIGURES
# ============================================================

plt.figure(figsize=(10,6))
for rep_name in agg["replication"].unique():
    sub = agg[agg["replication"] == rep_name]
    plt.plot(sub["K"], sub["reorganization_score_mean"], marker="o", label=rep_name)

plt.axvspan(0.70, 0.85, alpha=0.12, label="expected Kuramoto window 0.70–0.85")
plt.xlabel("K")
plt.ylabel("Reorganization score")
plt.title("Kuramoto minimal replication: reorganization score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_minimal_replication_reorganization_score.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for rep_name in agg["replication"].unique():
    sub = agg[agg["replication"] == rep_name]
    plt.plot(sub["K"], sub["switching_rate_mean"], marker="o", label=rep_name)

plt.xlabel("K")
plt.ylabel("Switching rate")
plt.title("Kuramoto minimal replication: switching rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_minimal_replication_switching_rate.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for rep_name in agg["replication"].unique():
    sub = agg[agg["replication"] == rep_name]
    plt.plot(sub["K"], sub["R_mean_mean"], marker="o", label=rep_name)

plt.xlabel("K")
plt.ylabel("R mean")
plt.title("Kuramoto minimal replication: synchronization curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_minimal_replication_R_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(summary["peak_K_reorganization"], summary["replication"], s=120)
plt.axvspan(0.70, 0.85, alpha=0.12)
plt.xlabel("Peak K")
plt.ylabel("Replication")
plt.title("Kuramoto minimal replication: peak K stability")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_kuramoto_minimal_replication_peak_K_summary.png", dpi=300)
plt.show()

# ============================================================
# DISPLAY
# ============================================================

print("\nSUMMARY")
display(summary)

print("\nREPLICATION WINDOW")
display(replication_window)

print("\nAGGREGATE PREVIEW")
display(agg.head(30))

print("\nSaved to:")
print(OUT)

In [ ]:
# ============================================================
# KURAMOTO — NULL / CONTROL + RECOVERY / PERTURBATION
# ============================================================

!pip install tqdm -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ============================================================
# PARAMETERS
# ============================================================

N = 400
dt = 0.03
steps = 2500
transient = 700

K_values = np.arange(0.5, 1.151, 0.05)

omega_std = 0.4
noise_std = 0.01

n_runs = 10

kick_strength = np.pi / 2
kick_fraction = 0.25

# ============================================================
# HELPERS
# ============================================================

def kuramoto_step(theta, omega, K):

    phase_diff = theta[:, None] - theta[None, :]
    coupling = np.mean(np.sin(-phase_diff), axis=1)

    dtheta = omega + K * coupling

    theta = theta + dt * dtheta

    return theta


def order_parameter(theta):

    return np.abs(np.mean(np.exp(1j * theta)))


def simulate_kuramoto(
    K,
    shuffled=False,
    recovery=False
):

    omega = np.random.normal(0, omega_std, N)

    if shuffled:
        np.random.shuffle(omega)

    theta = np.random.uniform(0, 2*np.pi, N)

    R_series = []

    kicked = False

    for step in range(steps):

        theta = kuramoto_step(theta, omega, K)

        theta += np.random.normal(0, noise_std, N)

        # ----------------------------------------------------
        # perturbation
        # ----------------------------------------------------

        if recovery and (step == 1400):

            idx = np.random.choice(
                N,
                int(kick_fraction * N),
                replace=False
            )

            theta[idx] += np.random.uniform(
                -kick_strength,
                kick_strength,
                len(idx)
            )

            kicked = True

        # ----------------------------------------------------

        if step > transient:

            R_series.append(order_parameter(theta))

    R_series = np.array(R_series)

    # --------------------------------------------------------
    # metrics
    # --------------------------------------------------------

    R_mean = np.mean(R_series)

    R_std = np.std(R_series)

    switching_rate = np.mean(
        np.abs(np.diff(R_series))
    )

    reorganization_score = (
        R_std * switching_rate * (1 - R_mean)
    )

    # --------------------------------------------------------
    # recovery metric
    # --------------------------------------------------------

    recovery_score = np.nan

    if recovery:

        half = len(R_series) // 2

        pre = np.mean(R_series[:half])

        post = np.mean(R_series[-half:])

        recovery_score = 1 - abs(pre - post)

    return {
        "R_mean": R_mean,
        "R_std": R_std,
        "switching_rate": switching_rate,
        "reorganization_score": reorganization_score,
        "recovery_score": recovery_score
    }


# ============================================================
# MAIN LOOP
# ============================================================

all_results = []

conditions = [
    "original",
    "shuffled_control",
    "recovery_test"
]

for condition in conditions:

    print("\n" + "="*60)
    print("CONDITION:", condition)
    print("="*60)

    for K in K_values:

        print(f"\nK={K:.2f}")

        for run in tqdm(range(n_runs)):

            shuffled = (
                condition == "shuffled_control"
            )

            recovery = (
                condition == "recovery_test"
            )

            res = simulate_kuramoto(
                K=K,
                shuffled=shuffled,
                recovery=recovery
            )

            res["condition"] = condition
            res["K"] = K

            all_results.append(res)

# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(all_results)

summary = (
    df
    .groupby(["condition", "K"])
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"],
        "switching_rate": ["mean", "std"],
        "reorganization_score": ["mean", "std"],
        "recovery_score": ["mean", "std"]
    })
)

summary.columns = [
    "_".join(col)
    for col in summary.columns
]

summary = summary.reset_index()

# ============================================================
# PEAKS
# ============================================================

peak_rows = []

for condition in conditions:

    sub = summary[
        summary["condition"] == condition
    ]

    idx = sub["reorganization_score_mean"].idxmax()

    peak_rows.append({
        "condition": condition,
        "peak_K": sub.loc[idx, "K"],
        "peak_reorganization": sub.loc[idx, "reorganization_score_mean"],
        "peak_R_mean": sub.loc[idx, "R_mean_mean"],
        "peak_switching": sub.loc[idx, "switching_rate_mean"],
        "peak_recovery": sub.loc[idx, "recovery_score_mean"]
    })

peak_df = pd.DataFrame(peak_rows)

# ============================================================
# PLOTS
# ============================================================

plt.figure(figsize=(8,5))

for condition in conditions:

    sub = summary[
        summary["condition"] == condition
    ]

    plt.plot(
        sub["K"],
        sub["reorganization_score_mean"],
        marker="o",
        label=condition
    )

plt.xlabel("K")
plt.ylabel("Reorganization score")
plt.title("Kuramoto: null/control + recovery")
plt.legend()
plt.grid(True)
plt.show()

# ------------------------------------------------------------

plt.figure(figsize=(8,5))

for condition in conditions:

    sub = summary[
        summary["condition"] == condition
    ]

    plt.plot(
        sub["K"],
        sub["switching_rate_mean"],
        marker="o",
        label=condition
    )

plt.xlabel("K")
plt.ylabel("Switching rate")
plt.title("Kuramoto: switching")
plt.legend()
plt.grid(True)
plt.show()

# ------------------------------------------------------------

plt.figure(figsize=(8,5))

for condition in conditions:

    sub = summary[
        summary["condition"] == condition
    ]

    plt.plot(
        sub["K"],
        sub["R_mean_mean"],
        marker="o",
        label=condition
    )

plt.xlabel("K")
plt.ylabel("Mean synchronization")
plt.title("Kuramoto: synchronization")
plt.legend()
plt.grid(True)
plt.show()

# ------------------------------------------------------------

plt.figure(figsize=(8,5))

sub = summary[
    summary["condition"] == "recovery_test"
]

plt.plot(
    sub["K"],
    sub["recovery_score_mean"],
    marker="o"
)

plt.xlabel("K")
plt.ylabel("Recovery score")
plt.title("Kuramoto recovery after perturbation")
plt.grid(True)
plt.show()

# ============================================================
# OUTPUT
# ============================================================

print("\nFULL LOOP DONE\n")

print("PEAK SUMMARY\n")
print(peak_df)

print("\nAGGREGATE PREVIEW\n")
print(summary.head(20))